In [2]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import load_yaml_config
from src.data import download_data, load_data_from_json
from src.plotting import plot_C_24

config = load_yaml_config(project_root / "config.yaml")
city = "Lisboa"

In [4]:
# Download data if it isn't already downloaded
download_data(city, config)

HTTP status: 200
Request URL: https://archive-api.open-meteo.com/v1/archive?latitude=38.7223&longitude=-9.1393&start_date=2020-01-01&end_date=2025-12-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Cwind_speed_10m&timezone=Europe%2FLisbon


In [5]:
# Load data into a DataFrame
df = load_data_from_json(city, config)

In [6]:
# Get the information of all data types present in the DataFrame
# With 52608 entries, we can check if there any null values in any existing column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  52608 non-null  datetime64[us]
 1   temperature_2m        52608 non-null  float64       
 2   relative_humidity_2m  52608 non-null  int64         
 3   precipitation         52608 non-null  float64       
 4   wind_speed_10m        52608 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1)
memory usage: 2.0 MB


In [7]:
df.describe()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
count,52608,52608.000000,52608.000000,52608.000000,52608.000000
mean,2022-12-31 23:30:00,17.275540,74.434877,0.074584,12.731623
min,2020-01-01 00:00:00,1.500000,14.000000,0.000000,0.000000
25%,2021-07-01 23:45:00,13.600000,63.000000,0.000000,7.800000
50%,2022-12-31 23:30:00,16.900000,78.000000,0.000000,12.000000
75%,2024-07-01 23:15:00,20.200000,89.000000,0.000000,16.700000
max,2025-12-31 23:00:00,40.700000,100.000000,19.200000,48.800000
std,NaN,5.285831,17.044434,0.437635,6.400850


In [8]:
# Check basic duplicated data
df.duplicated().sum()

np.int64(0)

In [9]:
# Create the rain event
df["rain_event"] = (df["precipitation"] >= 0.1).astype(int)

# Count the occurences of rain event
df["rain_event"].value_counts()

rain_event
0    46260
1     6348
Name: count, dtype: int64

In [10]:
# Check percentage of expected rain event
df["rain_event"].value_counts(normalize=True)

rain_event
0    0.879334
1    0.120666
Name: proportion, dtype: float64

In [11]:
def check_hourly_continuity(df):
    timestamps = (
        pd.to_datetime(df["time"])
        .sort_values()
        .reset_index(drop=True)
    )
    
    differences = timestamps.diff()
    
    irregular = differences[
        differences.notna() &
        (differences != pd.Timedelta(hours=1))
    ]
    
    print(f"Number of observations: {len(timestamps)}")
    print(f"Duplicate timestamps: {timestamps.duplicated().sum()}")
    print(f"Non-hourly transistions: {len(irregular)}")
    
    if irregular.empty:
        print("The stream is continuous at hourly intervals.")
    else:
        print("\nIrregular transitions:")
        for i in irregular.index:
            print(
                f"{timestamps.iloc[i-1]} ->"
                f"{timestamps.iloc[i]} "
                f"[{differences.iloc[i]}]"
            )

check_hourly_continuity(df)

Number of observations: 52608
Duplicate timestamps: 0
Non-hourly transistions: 0
The stream is continuous at hourly intervals.


In [12]:
# Check the humidity where rain_events  == 1
df_rain_events = df[df["rain_event"] == 1]

df_rain_events.describe()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,rain_event
count,6348,6348.00000,6348.000000,6348.000000,6348.000000,6348.0
mean,2023-02-15 04:28:16.219281,16.22678,84.485192,0.618100,17.213815,1.0
min,2020-01-02 19:00:00,7.10000,40.000000,0.100000,0.400000,1.0
25%,2021-04-28 14:45:00,14.10000,78.000000,0.100000,11.775000,1.0
50%,2023-01-19 22:30:00,16.00000,88.000000,0.200000,16.700000,1.0
75%,2024-10-11 18:15:00,18.40000,93.000000,0.600000,22.400000,1.0
max,2025-12-28 03:00:00,29.90000,100.000000,19.200000,48.800000,1.0
std,NaN,3.18714,10.510355,1.118679,7.627785,0.0


In [13]:
proportion_rain_above_50_humidity = (
    df_rain_events["relative_humidity_2m"] >= 50
).mean()

print(proportion_rain_above_50_humidity)

0.998109640831758


In [14]:
plot_C_24(config, df, city)

Saved /home/yoru/Documents/novaims/Y1/S1/DSM/Practical_Assessments/Data-Stream-Mining-Practical-Assignment-1/temp/Lisboa_C_24_iwgwrgty.pdf


In [21]:
c24 = df["rainy_hours_previous_24h"].dropna()

c24.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

count    52584.000000
mean         2.897307
std          4.964552
min          0.000000
25%          0.000000
50%          0.000000
75%          4.000000
90%         10.000000
95%         15.000000
99%         21.000000
max         24.000000
Name: rainy_hours_previous_24h, dtype: float64